In [ ]:
load_ext jupyter_black

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.multivariate.multivariate_ols import _MultivariateOLS
import statsmodels.formula.api as smf

In [ ]:
questions = pd.read_pickle("data/questions.gz")
questions_answers = dict(zip(questions.q_id, questions.correct_answer))

In [ ]:
options = {
    "q_50": ["1", "2", "3", "4"],
    "q_51": ["A", "B"],
    "q_52": ["1", "2", "3"],
    "q_53": ["A", "B", "C"],
    "q_54": ["10", "2", "3", "4", "5", "6", "7", "8", "9", "1"],
    "q_55": ["10", "2", "3", "4", "5", "6", "7", "8", "9", "1"],
    "q_56": ["10", "2", "3", "4", "5", "6", "7", "8", "9", "1"],
    "q_57": ["1", "2", "3", "4"],
    "q_58": [
        "1,1",
        "1,2",
        "1,3",
        "1,4",
        "2,1",
        "2,2",
        "2,3",
        "2,4",
        "3,1",
        "3,2",
        "3,3",
        "3,4",
        "4,1",
        "4,2",
        "4,3",
        "4,4",
    ],
    "q_59": [
        "Good manners",
        "Independence",
        "Hard work",
        "Feeling of responsibility",
        "Imagination",
        "Tolerance and respect for other people",
        "Thrift, saving money and things",
        "Determination",
        "Religious faith",
        "Not being selfish",
        "Obedience",
    ],
}

id_col = {
    "prism": "conversation_id",
    "chen": "text_id",
    "cad_en": "conversation_id",
    "cad_fr": "conversation_id",
    "cad_pt": "conversation_id",
    "cad_it": "conversation_id",
}

demographics = {
    "prism": [
        "age",
        "gender",
        "employment_status",
        "education",
        "marital_status",
        "english_proficiency",
        "religion",
        "ethnicity",
        "birth_region",
        "reside_region",
        "lm_familiarity",
    ],
    "chen": [
        "Gender",
        "human_Gender",
    ],
    "cad_en": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_fr": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_pt": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_it": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
}

In [ ]:
for dataset in [
    "cad_en",  # "chen",
    "prism",
]:
    df = pd.read_pickle(f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_answers.gz")
    df = df.rename(columns={"label": "Gender"})
    for c in [f"q_{i}" for i in range(50)]:
        df[c] = df[c].str.lower() == questions_answers[c]

    df["accuracy"] = df[[f"q_{i}" for i in range(50)]].mean(axis=1) * 100

    for c in [
        "q_50",
        "q_51",
        "q_52",
        "q_53",
        "q_54",
        "q_55",
        "q_56",
        "q_57",
        "q_58",
        "q_59",
    ]:
        if c == "q_53":
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].replace({"A": 0, "B": 0.5, "C": 1})
            df[c] = df[c].astype(float)
        elif c == "q_51":
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].replace({"A": 0, "B": 1})
            df[c] = df[c].astype(float)
        elif c == "q_58":
            df[c] = df[c].str.replace(" ", "")
            df[c] = df[c].replace({"1,1":0, "1,3":0, "3,1":0, "3,3": 0, "2,2": 1, "1,2": 0.5, "1,4": 0.5, "2,3": 0.5, "2,1": 0.5, "3,2": 0.5, "3,4": 0.5, "4,1": 0.5, "4,3": 0.5, "2,4": 1, "4,2": 1, "4,4": 1})
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
        elif c == "q_59":
            for v in options[c]:
                df[f"{c}_{v}"] = ~df[c].str.extract("(" + v + ")").isna()
        else:
            df[c] = df[c].str.extract(f"({'|'.join(options[c])})")
            df[c] = df[c].astype(float)
    df = df.drop(columns=[f"q_{i}" for i in range(50)] + ["q_59", "q_60"])

    df_linguistic = pd.read_pickle(
        f"data/{dataset + '_utterances' if dataset != 'chen' else dataset}_linguistic.gz"
    )
    for c in ["politeness_user_prompt", "politeness_model_response"]:
        if c in df_linguistic:
            df_linguistic[c] = df_linguistic[c].replace(
                {"impolite": 0, "neutral": 0.5, "polite": 1, "somewhat polite": 0.75}
            )
    df_linguistic = df_linguistic.rename(columns={"gpt_description": "topic"})
    if dataset != "chen":
        if dataset == "prism":
            demographics[dataset] += ["model_name"]
        demographics[dataset] += ["topic"]
        group_cols = [id_col[dataset]] + demographics[dataset]
        df_linguistic = (
            df_linguistic.groupby(group_cols)[
                [
                    c
                    for c in df_linguistic.columns
                    if "_model_response" in c or "_user_prompt" in c
                ]
            ]
            .mean()
            .reset_index()
        )
    df = df.merge(
        df_linguistic[
            [id_col[dataset]]
            + [
                c
                for c in df_linguistic.columns
                if "_model_response" in c
                or "_user_prompt" in c
                or c == "topic"
                or c == "model_name"
            ]
        ],
        on=id_col[dataset],
    )
    demographics[dataset] += [
        c for c in df.columns if "_model_response" in c or "_user_prompt" in c
    ]

    if "topic" in df:
        topic_dict = {
            df["topic"].unique()[i]: str(i) for i in range(len(df["topic"].unique()))
        }
        reverse_topic_dict = {f"topic[T.{topic_dict[topic]}]": topic for topic in topic_dict}
        df["topic"] = df["topic"].replace(topic_dict)

    df_beliefs = pd.read_pickle(
        f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_beliefs_preprocessed.gz"
    )
    df_beliefs.columns = df_beliefs.columns.str.replace(" ", "")
    cols = [
        c
        for c in df_beliefs.columns
        if "shared_extracted_" in c or "value_JSON_" in c or "unknown_token_" in c
    ] + ["revealed_Gender"]
    if "human_Gender" in df_beliefs.columns:
        cols += ["human_Gender"]
    demographics[dataset] += cols
    cols.append(id_col[dataset])
    df = df.merge(df_beliefs[cols], on=id_col[dataset])

    for col in (
        [
            "accuracy",
            "q_50",
            "q_51",
            "q_52",
            "q_53",
            "q_54",
            "q_55",
            "q_56",
            "q_57",
            "q_58",
        ]
        + [f"q_59_{v}" for v in options["q_59"] if v in df]
    ):
        filtered_df = df.loc[~df[col].isna()]
        demo_cols = [c for c in demographics[dataset] if "_model_response" in c
                or "_user_prompt" in c
                or c == "model_name"]
        mod = smf.ols(formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df)
        res = mod.fit()
        result_df = pd.read_html(res.summary().tables[1].as_html(),header=0,index_col=0)[0].reset_index()
        fig = plt.figure(figsize=(6.5,5))
        ax = sns.barplot(result_df.loc[(result_df['P>|t|']<0.05)&(result_df['index']!='Intercept')].sort_values(by='coef'), x='index', y='coef')
        ax.tick_params(axis='x', labelrotation=90)
        fig.savefig(f'figures_regression/{dataset}_{col}_1.png', bbox_inches='tight')
        plt.show()
        print(res.summary())
        demo_cols = [
                c
                for c in demographics[dataset]
               if "_model_response" in c
                or "_user_prompt" in c
                or c == "topic"
                or c == "model_name"
        ]
        mod = smf.ols(formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df)
        res = mod.fit()
        result_df = pd.read_html(res.summary().tables[1].as_html(),header=0,index_col=0)[0].reset_index()
        result_df['linguistic'] = ~result_df['index'].str.contains('topic')
        result_df['index'] = result_df['index'].replace(reverse_topic_dict)
        fig = plt.figure(figsize=(40,5))
        ax = sns.barplot(result_df.loc[(result_df['P>|t|']<0.05)&(result_df['index']!='Intercept')].sort_values(by='coef'), x='index', y='coef', hue='linguistic')
        ax.tick_params(axis='x', labelrotation=90)
        fig.savefig(f'figures_regression/{dataset}_{col}_2.png', bbox_inches='tight')
        plt.show()
        print(res.summary())

        for demographic in [
            "age",
            "gender",
            "education",
            "ethnicity",
            "religion",
            "english",
            "marital",
        ]:
            demo_cols = [
                c
                for c in demographics[dataset]
                if demographic in c.lower()
                or "_model_response" in c
                or "_user_prompt" in c
                or c == "topic"
                or c == "model_name"
            ]
            mod = smf.ols(formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df)
            res = mod.fit()
            result_df = pd.read_html(res.summary().tables[1].as_html(),header=0,index_col=0)[0].reset_index()
            result_df['type'] = result_df['index'].str.extract('(topic)')
            result_df['type'].loc[result_df['index'].str.contains(demographic)]='demographic'
            result_df['type'].loc[result_df['type'].isna()]= 'linguistic'
            result_df['index'] = result_df['index'].replace(reverse_topic_dict)
            fig = plt.figure(figsize=(45,5))
            ax = sns.barplot(result_df.loc[(result_df['P>|t|']<0.05)&(result_df['index']!='Intercept')].sort_values(by='coef'), x='index', y='coef', hue='type')
            ax.tick_params(axis='x', labelrotation=90)
            fig.savefig(f'figures_regression/{dataset}_{col}_{demographic}_3.png', bbox_inches='tight')
            plt.show()
            print(res.summary())

            
        # for demographic in [
        #     "age",
        #     "gender",
        #     "education",
        #     "ethnicity",
        #     "religion",
        #     "english",
        #     "marital",
        # ]:
        #     demo_cols = [c for c in demographics[dataset] if demographic in c.lower()]
        #     mod = smf.ols(formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df)
        #     res = mod.fit()
        #     print(res.summary())
        #     demo_cols = [
        #         c
        #         for c in demographics[dataset]
        #         if demographic in c.lower()
        #         or "_model_response" in c
        #         or "_user_prompt" in c
        #         or c == "model_name"
        #     ]
        #     mod = smf.ols(formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df)
        #     res = mod.fit()
        #     print(res.summary())
        #     demo_cols = [
        #         c
        #         for c in demographics[dataset]
        #         if demographic in c.lower()
        #         or "_model_response" in c
        #         or "_user_prompt" in c
        #         or c == "topic"
        #         or c == "model_name"
        #     ]
        #     mod = smf.ols(formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df)
        #     res = mod.fit()
        #     print(res.summary())